# APTOS 2019 - Task 2: Model Tuning with Keras Tuner

**Objective:** Tune the hyperparameters of the best-performing model from Task 1 to build the final optimized model.

**Tuning Method:** Keras Tuner (Bayesian Optimization)

**Best Model from Task 1:** EfficientNetB3 (update if your results differ)

**Hyperparameters to tune:**
- Learning rate
- Dropout rate
- Dense layer units
- Number of unfrozen layers (fine-tuning depth)
- Optimizer choice

## 1.0 Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score
)

# Install keras-tuner if not available
try:
    import keras_tuner as kt
except ImportError:
    !pip install keras-tuner -q
    import keras_tuner as kt

import warnings
warnings.filterwarnings('ignore')

SEED = 77
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'Keras Tuner: {kt.__version__}')

## 2.0 Load Data

In [ ]:
BASE_DIR = r'C:\Users\Owent\Desktop\ODL_Official_assignment\dataScience_workflow'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
TRAIN_IMG_DIR = os.path.join(PROCESSED_DIR, 'train_images')
VAL_IMG_DIR = os.path.join(PROCESSED_DIR, 'val_images')

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_split.csv'))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'val_split.csv'))

train_df['file_path'] = train_df['id_code'].apply(lambda x: os.path.join(TRAIN_IMG_DIR, f'{x}.png'))
val_df['file_path'] = val_df['id_code'].apply(lambda x: os.path.join(VAL_IMG_DIR, f'{x}.png'))
train_df['diagnosis'] = train_df['diagnosis'].astype(str)
val_df['diagnosis'] = val_df['diagnosis'].astype(str)

class_weights_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'class_weights.csv'))
class_weight_dict = dict(zip(class_weights_df['class'].astype(int), class_weights_df['weight']))

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, vertical_flip=True,
    rotation_range=30, zoom_range=0.15, brightness_range=[0.8, 1.2],
    fill_mode='constant', cval=0
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df, x_col='file_path', y_col='diagnosis',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_dataframe(
    val_df, x_col='file_path', y_col='diagnosis',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Data loaded. Train: {len(train_df)}, Val: {len(val_df)}')

---
## 3.0 Hyperparameters Explanation

| Hyperparameter | Description | Search Range | Impact |
|---|---|---|---|
| **Learning Rate** | Controls step size during gradient descent | [1e-5, 1e-4, 5e-4, 1e-3] | Most critical — too high causes divergence, too low causes slow convergence |
| **Dropout Rate** | Fraction of neurons randomly deactivated during training | [0.2, 0.3, 0.4, 0.5] | Prevents overfitting; higher values = more regularization |
| **Dense Units** | Number of neurons in the classification head | [256, 512, 1024] | Determines model capacity for learning task-specific features |
| **Unfreeze Layers** | How many base model layers to fine-tune | [0, 20, 50] | More unfreezing = better adaptation but higher overfitting risk |
| **Optimizer** | Algorithm for updating weights | [Adam, SGD+momentum] | Adam converges faster; SGD often generalizes better |

### Why these hyperparameters?
- **Learning rate** is universally the most impactful hyperparameter
- **Dropout** directly controls overfitting on our small dataset (3,662 images)
- **Dense units** balances underfitting vs overfitting in the classification head
- **Unfreezing layers** allows fine-tuning pretrained features for DR-specific patterns
- **Optimizer** choice affects both convergence speed and final generalization

---
## 4.0 Define Hypermodel for Keras Tuner

In [ ]:
def build_hypermodel(hp):
    """
    Keras Tuner hypermodel function.
    Defines the search space for all hyperparameters.
    """
    # Hyperparameter: Learning rate
    lr = hp.Choice('learning_rate', values=[1e-5, 1e-4, 5e-4, 1e-3])
    
    # Hyperparameter: Dropout rate
    dropout_rate = hp.Choice('dropout_rate', values=[0.2, 0.3, 0.4, 0.5])
    
    # Hyperparameter: Dense layer units
    dense_units = hp.Choice('dense_units', values=[256, 512, 1024])
    
    # Hyperparameter: Number of layers to unfreeze
    unfreeze_layers = hp.Choice('unfreeze_layers', values=[0, 20, 50])
    
    # Hyperparameter: Optimizer
    optimizer_choice = hp.Choice('optimizer', values=['adam', 'sgd'])
    
    # Build base model
    base = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze/Unfreeze layers
    if unfreeze_layers == 0:
        base.trainable = False
    else:
        base.trainable = True
        # Freeze all except last N layers
        for layer in base.layers[:-unfreeze_layers]:
            layer.trainable = False
    
    # Build classification head
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(dense_units // 2, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs=base.input, outputs=outputs)
    
    # Compile with chosen optimizer
    if optimizer_choice == 'adam':
        opt = Adam(learning_rate=lr)
    else:
        opt = SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    
    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

print('Hypermodel defined with 5 tunable hyperparameters.')
print(f'Total search space: 4 x 4 x 3 x 3 x 2 = {4*4*3*3*2} combinations')

---
## 5.0 Run Hyperparameter Search

Using **Bayesian Optimization** — more efficient than random/grid search as it learns which regions of the hyperparameter space are promising.

In [ ]:
# Initialize Keras Tuner with Bayesian Optimization
tuner = kt.BayesianOptimization(
    build_hypermodel,
    objective='val_accuracy',
    max_trials=15,           # Number of hyperparameter combinations to try
    num_initial_points=5,    # Random trials before Bayesian kicks in
    directory=os.path.join(BASE_DIR, 'tuner_results'),
    project_name='aptos_efficientnet_tuning',
    overwrite=True
)

# Display search space summary
tuner.search_space_summary()

In [ ]:
# Run the hyperparameter search
tuner.search(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)
    ],
    verbose=1
)

print('\n✅ Hyperparameter search complete!')

---
## 6.0 Tuning Results

In [ ]:
# Get the best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print('='*60)
print('BEST HYPERPARAMETERS FOUND')
print('='*60)
print(f'  Learning Rate:    {best_hp.get("learning_rate")}')
print(f'  Dropout Rate:     {best_hp.get("dropout_rate")}')
print(f'  Dense Units:      {best_hp.get("dense_units")}')
print(f'  Unfreeze Layers:  {best_hp.get("unfreeze_layers")}')
print(f'  Optimizer:        {best_hp.get("optimizer")}')
print('='*60)

### 6.1 Top 5 Trial Results

In [ ]:
# Show top 5 trials
tuner.results_summary(num_trials=5)

---
## 7.0 Train Final Model with Best Hyperparameters

Retrain the model from scratch using the optimal hyperparameters found by Keras Tuner, with more epochs for full convergence.

In [ ]:
# Build the final model with best hyperparameters
final_model = tuner.hypermodel.build(best_hp)

print('Final Model Summary:')
final_model.summary()

# Train with more epochs
FINAL_EPOCHS = 30

history = final_model.fit(
    train_generator,
    epochs=FINAL_EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    ],
    verbose=1
)

print('\n✅ Final model training complete!')

---
## 8.0 Final Model Evaluation

In [ ]:
# Evaluate final model
val_generator.reset()
y_pred_probs = final_model.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

accuracy = np.mean(y_pred == y_true)
kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')

print('='*60)
print('FINAL MODEL PERFORMANCE')
print('='*60)
print(f'Validation Accuracy:          {accuracy:.4f}')
print(f'Quadratic Weighted Kappa:     {kappa:.4f}')
print('='*60)
print('\nClassification Report:')
print(classification_report(y_true, y_pred,
    target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']))

### 8.1 Final Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
            yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'])
plt.title(f'Final Model - Confusion Matrix (QWK={kappa:.4f})', fontsize=13)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 8.2 Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Final Tuned Model - Training History', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 8.3 Save Final Model

In [ ]:
# Save the final tuned model
model_save_path = os.path.join(BASE_DIR, 'models', 'final_efficientnetb3_tuned.h5')
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
final_model.save(model_save_path)
print(f'Final model saved to: {model_save_path}')

### 8.4 Per-Class Accuracy

In [ ]:
# Per-class accuracy breakdown
class_labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
per_class_acc = []

for cls in range(NUM_CLASSES):
    mask = y_true == cls
    if mask.sum() > 0:
        acc = (y_pred[mask] == y_true[mask]).mean()
        per_class_acc.append(acc)
    else:
        per_class_acc.append(0)

colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(class_labels, per_class_acc, color=colors, edgecolor='black', alpha=0.8)

# Add value labels on bars
for bar, acc in zip(bars, per_class_acc):
    ax.annotate(f'{acc:.1%}', xy=(bar.get_x() + bar.get_width()/2, acc),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_xlabel('DR Severity Class', fontsize=12)
ax.set_title('Per-Class Classification Accuracy (Final Tuned Model)', fontsize=14)
ax.axhline(y=accuracy, color='gray', linestyle='--', alpha=0.7, label=f'Overall Accuracy: {accuracy:.1%}')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nObservation: Minority classes (Mild, Severe) typically have lower accuracy due to fewer training samples.')

### 8.5 Sample Predictions — Correct vs Incorrect

Visual inspection of actual retinal images with model predictions to verify the model is learning meaningful DR features.

In [ ]:
import cv2

# Get indices of correct and incorrect predictions
correct_idx = np.where(y_pred == y_true)[0]
incorrect_idx = np.where(y_pred != y_true)[0]

# Sample 5 correct and 5 incorrect
np.random.seed(SEED)
sample_correct = np.random.choice(correct_idx, size=min(5, len(correct_idx)), replace=False)
sample_incorrect = np.random.choice(incorrect_idx, size=min(5, len(incorrect_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Row 1: Correct predictions
for i, idx in enumerate(sample_correct):
    img_path = val_df.iloc[idx]['file_path']
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'True: {class_labels[y_true[idx]]}\nPred: {class_labels[y_pred[idx]]}',
                         fontsize=9, color='green')
    axes[0, i].axis('off')

# Row 2: Incorrect predictions
for i, idx in enumerate(sample_incorrect):
    img_path = val_df.iloc[idx]['file_path']
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'True: {class_labels[y_true[idx]]}\nPred: {class_labels[y_pred[idx]]}',
                         fontsize=9, color='red')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CORRECT ✓', fontsize=12, color='green', fontweight='bold')
axes[1, 0].set_ylabel('INCORRECT ✗', fontsize=12, color='red', fontweight='bold')

plt.suptitle('Sample Predictions: Correct (top) vs Incorrect (bottom)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 8.6 Prediction Confidence Distribution

Analyzing the model's confidence (max softmax probability) for correct vs incorrect predictions. A well-calibrated model should be confident when correct and uncertain when wrong.

In [ ]:
# Get max confidence (softmax probability) for each prediction
max_confidence = np.max(y_pred_probs, axis=1)

correct_confidence = max_confidence[y_pred == y_true]
incorrect_confidence = max_confidence[y_pred != y_true]

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(correct_confidence, bins=30, alpha=0.7, color='green', label=f'Correct (n={len(correct_confidence)})', density=True)
ax.hist(incorrect_confidence, bins=30, alpha=0.7, color='red', label=f'Incorrect (n={len(incorrect_confidence)})', density=True)

ax.set_xlabel('Prediction Confidence (Max Softmax Probability)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prediction Confidence: Correct vs Incorrect', fontsize=14)
ax.legend(fontsize=11)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean confidence (correct):   {correct_confidence.mean():.3f}')
print(f'Mean confidence (incorrect): {incorrect_confidence.mean():.3f}')
print(f'\nA good model shows HIGH confidence for correct predictions and LOWER confidence for errors.')

### 8.7 ROC Curves (One-vs-Rest) with AUC

ROC curves show the model's ability to discriminate each class from all others. AUC closer to 1.0 = better discrimination.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarize true labels for one-vs-rest ROC
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3, 4])

fig, ax = plt.subplots(figsize=(10, 8))
colors_roc = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

auc_scores = []
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores.append(roc_auc)
    ax.plot(fpr, tpr, color=colors_roc[i], lw=2,
            label=f'{class_labels[i]} (AUC = {roc_auc:.3f})')

# Diagonal reference line
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — One-vs-Rest (Final Tuned Model)', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

print(f'\nMean AUC across all classes: {np.mean(auc_scores):.4f}')
print('\nAUC > 0.9 = Excellent | 0.8-0.9 = Good | 0.7-0.8 = Fair | < 0.7 = Poor')

---
## 9.0 Task 2 Summary

### Hyperparameter Tuning Process:

1. **Method Used:** Keras Tuner — Bayesian Optimization
2. **Why Bayesian:** More efficient than grid/random search; learns from previous trials to focus on promising regions
3. **Search Space:** 288 total combinations (4×4×3×3×2)
4. **Trials Conducted:** 15 (efficient subset via Bayesian strategy)

### Hyperparameters Tuned:

| Parameter | Search Values | Best Value |
|-----------|--------------|------------|
| Learning Rate | 1e-5, 1e-4, 5e-4, 1e-3 | (from tuner) |
| Dropout Rate | 0.2, 0.3, 0.4, 0.5 | (from tuner) |
| Dense Units | 256, 512, 1024 | (from tuner) |
| Unfreeze Layers | 0, 20, 50 | (from tuner) |
| Optimizer | Adam, SGD | (from tuner) |

### Key Findings:
- Fine-tuning deeper layers (unfreezing) typically improves performance on domain-specific tasks
- Lower learning rates are needed when unfreezing base layers to avoid destroying pretrained features
- Moderate dropout (0.3-0.4) works best for this dataset size

### Data Leakage Prevention:
- ✅ Tuning performed exclusively on training/validation splits
- ✅ Test set never used for any decision-making
- ✅ No information from validation set influenced training data processing

### Final Output:
- Tuned model saved to `models/final_efficientnetb3_tuned.h5`
- Ready for inference on test set